In [2]:
import os
import json
import openai
from tqdm import tqdm

# 1. Configure API key (ensure OPENAI_API_KEY is set in your environment)
openai.api_key = os.getenv("OPENAI_API_KEY")

# 2. Load your tags
with open('tags.json') as f:
    tags = json.load(f)

# 3. Load draft and model texts
with open('protected/Draft 2.md') as f:
    draft2 = f.read()

with open('protected/the-combinatorial-model.md') as f:
    combinatorial_model = f.read()

# 4. Define a function to refine tag descriptions
def refine_tag_description(tag_id, description, draft_text, model_text):
    prompt = f"""
You are an expert in reading research and cognitive science. 
Given the following tag identifier and its current description, refine and elaborate the description to be more precise, informative, and aligned with the themes and arguments in the provided documents.
Return only the refined description as a single concise paragraph.

Tag ID: {tag_id}
Current Description: {description}

Context from Draft 2:
\"\"\"{draft_text}\"\"\"

Context from The Combinatorial Model:
\"\"\"{model_text}\"\"\"

Please focus on ensuring the description captures the theoretical nuances and relevance to the article draft.
"""
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You refine and elaborate tag descriptions for scholarly research."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=250
    )
    return response.choices[0].message.content.strip()

# 5. Iterate and refine all tags
refined_tags = []
for tag in tqdm(tags):
    refined = refine_tag_description(tag["id"], tag["description"], draft2, combinatorial_model)
    refined_tags.append({
        "id": tag["id"],
        "primaryWeight": tag["primaryWeight"],
        "description": refined
    })

# 6. Save to JSON
with open('refined_tags.json', 'w') as f:
    json.dump(refined_tags, f, indent=2)

print("Refined tags saved to refined_tags.json")


100%|██████████| 24/24 [09:41<00:00, 24.24s/it]

Refined tags saved to refined_tags.json
